In [ ]:


def apply_vat_calculation(
    df: pd.DataFrame,
    cost_df: pd.DataFrame | None = None,
    item_col: str = '품목코드',
    qty_col: str = '수량',
    unit_price_col: str = '단가(vat포함)',
    unit_price_ex_vat_col: str = '단가',   # VAT미포함 단가 컬럼명(=이카운트 "단가")
    tax_flag_col_candidates=('과세/면세', '과세여부', 'TAX_YN', 'TAX_FLAG', 'VAT_YN'),
):
    """
    품목별 과세/면세를 반영한 공급가액/부가세/합계 계산 + 단가(VAT미포함) 컬럼 생성

    [과세]
      합계 = 수량 * 단가(vat포함)
      공급가액 = 합계 / 1.1
      부가세 = 합계 - 공급가액
      단가(미포함) = 단가(vat포함) / 1.1

    [면세]
      공급가액 = 수량 * 단가(vat포함)
      부가세 = 0
      합계 = 공급가액
      단가(미포함) = 단가(vat포함)
    """
    if qty_col not in df.columns or unit_price_col not in df.columns:
        return df

    qty = _to_number(df[qty_col]).fillna(0)
    price_vat = _to_number(df[unit_price_col]).fillna(0)
    total = qty * price_vat

    supply = total / 1.1
    vat = total - supply
    unit_price_ex_vat = price_vat / 1.1

    # cost_df로 과세/면세 판단해서 행별로 덮어쓰기
    if cost_df is not None and item_col in df.columns and item_col in cost_df.columns:
        tax_col = None
        for c in tax_flag_col_candidates:
            if c in cost_df.columns:
                tax_col = c
                break

        if tax_col is not None:
            tmp = cost_df[[item_col, tax_col]].copy()
            tmp[item_col] = tmp[item_col].astype(str).str.strip()

            t = tmp[tax_col].astype(str).str.strip().str.upper()
            tmp['_is_exempt'] = (
                t.str.contains('면세', na=False) |
                t.isin(['N', 'NO', '0', 'FALSE', 'F'])
            )

            df_item = df[item_col].astype(str).str.strip()
            is_exempt = df_item.map(
                tmp.drop_duplicates(item_col).set_index(item_col)['_is_exempt']
            ).fillna(False)

            # 면세 행: 공급가액=합계, 부가세=0
            supply = supply.where(~is_exempt, total)
            vat = vat.where(~is_exempt, 0)

            # 면세 행: 단가(미포함)=단가(vat포함)
            unit_price_ex_vat = unit_price_ex_vat.where(~is_exempt, price_vat)

    df['합계'] = total.round(0)
    df['공급가액'] = supply.round(0)
    df['부가세'] = vat.round(0)
    df[unit_price_ex_vat_col] = unit_price_ex_vat.round(0)

    return df


def append_price_check_note(
    df: pd.DataFrame,
    company_col: str = "거래처명",
    price_col: str = "단가(vat포함)",
    memo_col: str = "적요",
):
    """
    '주식회사 꿈꾸는사람들' 거래처에 대해
    단가(vat포함) 1의 자리가 0이 아니면 적요에 '단가 확인 필요' 추가
    """
    if company_col not in df.columns or price_col not in df.columns:
        return df, 0

    if memo_col not in df.columns:
        df[memo_col] = ""

    comp = df[company_col].astype(str).str.strip()
    price = pd.to_numeric(df[price_col], errors="coerce")
    price = price.replace([np.inf, -np.inf], np.nan)

    mask = (comp == DREAM_COMPANY) & price.notna() & ((np.floor(price + 0.5) % 10) != 0)


    if mask.any():
        cur = df.loc[mask, memo_col].astype(str).fillna("")
        df.loc[mask, memo_col] = cur.apply(
            lambda s: s if PRICE_CHECK_NOTE in s else ((s + " " if s.strip() else "") + PRICE_CHECK_NOTE)
        )

    return df, int(mask.sum())